<a href="https://colab.research.google.com/github/mf2056/Dissertation/blob/main/InceptionV3%2BRF(Bayesian).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 2.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer

import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import Input, Conv2D, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

In [4]:
X_train = np.load('/content/drive/MyDrive/X_train.npy')
Y_train_lab = np.load('/content/drive/MyDrive/Y_train_labels.npy')
X_test = np.load('/content/drive/MyDrive/X_test.npy')
Y_test_lab = np.load('/content/drive/MyDrive/Y_test_labels.npy')

In [5]:
print("--- LABEL MAPPING KEY ---")

class_map = {
    10: ("Tree cover", "#006400"),
    20: ("Shrubland", "#ffbb22"),
    30: ("Grassland", "#ffff4c"),
    40: ("Cropland", "#f096ff"),
    50: ("Built-up", "#fa0000"),
    60: ("Bare / Sparse vegetation", "#b4b4b4"),
    70: ("Snow and ice", "#f0f0f0"),
    80: ("Permanent water bodies", "#0064ff"),
    90: ("Herbaceous wetland", "#0096a0"),
}

# Create the internal mapping
unique_labels = sorted(np.unique(Y_train_lab))
label_map = {old: new for new, old in enumerate(unique_labels)}

# Sorting by the new index (0, 1, 2...) for readability
for old_id, new_id in sorted(label_map.items(), key=lambda item: item[1]):
    class_name = class_map.get(old_id, ("Unknown", ""))[0]
    print(f"New ID: {new_id}  <--  Original ID: {old_id} ({class_name})")

# Apply the mapping to create the final training/testing labels
Y_train_ready = np.array([label_map[l] for l in Y_train_lab])
Y_test_ready = np.array([label_map[l] for l in Y_test_lab])


print("\n--- DATA SHAPE VERIFICATION ---")
print(f"X_train: {X_train.shape}")
print(f"Y_train_ready: {Y_train_ready.shape}")
print(f"Unique classes in Training: {np.unique(Y_train_ready)}")

--- LABEL MAPPING KEY ---
New ID: 0  <--  Original ID: 10 (Tree cover)
New ID: 1  <--  Original ID: 20 (Shrubland)
New ID: 2  <--  Original ID: 30 (Grassland)
New ID: 3  <--  Original ID: 40 (Cropland)
New ID: 4  <--  Original ID: 50 (Built-up)
New ID: 5  <--  Original ID: 80 (Permanent water bodies)

--- DATA SHAPE VERIFICATION ---
X_train: (639, 256, 256, 7)
Y_train_ready: (639,)
Unique classes in Training: [0 1 2 3 4 5]


In [6]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
import numpy as np

# The Input (7 bands)
inputs = Input(shape=(256, 256, 7))

# The Adapter (7 -> 3 channels)
adapter = Conv2D(3, (1, 1), padding='same', name='band_adapter')(inputs)

# Load InceptionV3
inception_base = tf.keras.applications.InceptionV3(
    include_top=False,
    weights='imagenet',
    input_shape=(256, 256, 3)
)

# Construct the Temporary Training Model
x = inception_base(adapter)
pooled_output = GlobalAveragePooling2D(name='feature_layer')(x)
x = Dense(256, activation='relu')(pooled_output)
x = Dropout(0.5)(x)
temp_predictions = Dense(len(np.unique(Y_train_ready)), activation='softmax')(x)

# This model is for Fine-Tuning the Inception weights
trainable_model = Model(inputs=inputs, outputs=temp_predictions)

# This model is for Extracting features later
feature_extractor = Model(inputs=inputs, outputs=pooled_output)

print("Dynamic Hybrid Architecture defined.")

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Dynamic Hybrid Architecture defined.


In [7]:
from sklearn.utils.class_weight import compute_class_weight

# Calculate weights for class balance
weights = compute_class_weight('balanced', classes=np.unique(Y_train_ready), y=Y_train_ready)
class_weight_dict = dict(enumerate(weights))

# Phase 1 Freeze Inception, Train Adapter)
inception_base.trainable = False
trainable_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Starting Phase 1: Warming up Adapter and Head...")
trainable_model.fit(X_train, Y_train_ready, epochs=5,
                    batch_size=16,
                    validation_data=(X_test, Y_test_ready),
                    class_weight=class_weight_dict)


Starting Phase 1: Warming up Adapter and Head...
Epoch 1/5


KeyboardInterrupt: 

In [ ]:
# Phase 2: Fine-Tuning
inception_base.trainable = True
trainable_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # Low LR to protect patterns
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nStarting Phase 2: Fine-tuning Inception for Satellite Data...")
trainable_model.fit(X_train, Y_train_ready,
                    epochs=10,
                    batch_size=16,
                    validation_data=(X_test, Y_test_ready),
                    class_weight=class_weight_dict)

In [ ]:
print("Extracting dynamic training features...")
X_train_features = feature_extractor.predict(X_train, batch_size=16)

print("Extracting dynamic testing features...")
X_test_features = feature_extractor.predict(X_test, batch_size=16)

print(f"Dynamic Features Ready! Shape: {X_train_features.shape}")

In [8]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def on_step(optim_result):
    it = len(optim_result.x_iters)
    current_acc = -optim_result.func_vals[-1]
    print(f"{it:<10} | {current_acc:<10.4f} ")


In [ ]:
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical
from sklearn.ensemble import RandomForestClassifier

# Define the Search Space
search_space = {
    'n_estimators': Integer(50, 500),
    'max_depth': Integer(5, 50),
    'min_samples_split': Integer(2, 10),
    'max_features': Categorical(['sqrt', 'log2'])
}

rf = RandomForestClassifier(random_state=20, class_weight='balanced', n_jobs=-1)

opt = BayesSearchCV(
    estimator=rf,
    search_spaces=search_space,
    n_iter=20,
    cv=3,
    n_jobs=-1,
    verbose=0,
    random_state=20
)

print(f"\n{'Iteration':<10} | {'Curr Acc':<10}")
print("-" * 40)
opt.fit(X_train_features, Y_train_ready, callback=on_step)

# Results
print(f"\nBest Parameters found: {opt.best_params_}")
print(f"Best Validation Accuracy: {opt.best_score_:.4f}")

In [10]:
# from skopt import BayesSearchCV
# from sklearn.ensemble import RandomForestClassifier

# # 1. Define the Search Space
# # We are telling the optimizer to find the best values within these ranges
# search_space = {
#     'n_estimators': Integer(50, 500),         # Number of trees
#     'max_depth': Integer(5, 20),              # How deep each tree goes
#     'min_samples_leaf': Integer(5, 20),
#     'min_samples_split': Integer(5, 50),       # Minimum samples to split a node
#     'max_features': Categorical(['sqrt', 'log2']) # How many features to look at per split
# }

# # 2. Initialize the Random Forest
# rf = RandomForestClassifier(random_state=20, class_weight='balanced')

# # 3. Initialize Bayesian Search
# # n_iter=20 means it will try 20 different "smart" combinations
# opt = BayesSearchCV(
#     estimator=rf,
#     search_spaces=search_space,
#     n_iter=10,
#     cv=3,
#     n_jobs=-1,
#     verbose=0,
#     random_state=20
# )

# # 4. Run the Optimization
# # 1. Define the Table Header
# print(f"\n{'Iteration':<10} | {'Val Acc':<10} ")
# print("-" * 50)
# opt.fit(X_train_features, Y_train_ready, callback=on_step)

# # 5. Results
# print(f"\nBest Parameters found: {opt.best_params_}")
# print(f"Best Validation Accuracy: {opt.best_score_:.4f}")


Iteration  | Val Acc    
--------------------------------------------------
1          | 0.6854     
2          | 0.6980     
3          | 0.6823     
4          | 0.6917     
5          | 0.6870     
6          | 0.7042     
7          | 0.6604     
8          | 0.6964     
9          | 0.6714     
10         | 0.6870     

Best Parameters found: OrderedDict({'max_depth': 19, 'max_features': 'sqrt', 'min_samples_leaf': 9, 'min_samples_split': 48, 'n_estimators': 380})
Best Validation Accuracy: 0.7042


In [11]:
best_rf = opt.best_estimator_
y_pred = best_rf.predict(X_test_features)

target_names = ['Tree cover', 'Shrubland', 'Grassland', 'Cropland', 'Built-up', 'Permanent water']

# Final predictions
train_preds = best_rf.predict(X_train_features)
test_preds = best_rf.predict(X_test_features)

# Calculate metrics
train_acc = accuracy_score(Y_train_ready, train_preds)
test_acc = accuracy_score(Y_test_ready, test_preds)
test_f1 = f1_score(Y_test_ready, test_preds, average='macro')

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Test Macro F1:  {test_f1:.4f}")

print(classification_report(Y_test_ready, y_pred, target_names=target_names))

Train Accuracy: 0.9296
Test Accuracy:  0.6400
Test Macro F1:  0.5989
                 precision    recall  f1-score   support

     Tree cover       0.38      0.83      0.53         6
      Shrubland       0.50      0.46      0.48        13
      Grassland       0.60      0.21      0.31        29
       Cropland       0.69      0.86      0.76        28
       Built-up       0.54      0.86      0.67        22
Permanent water       1.00      0.74      0.85        27

       accuracy                           0.64       125
      macro avg       0.62      0.66      0.60       125
   weighted avg       0.67      0.64      0.62       125

